In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix
from sklearn.base import clone
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("All imports successful.")

All imports successful.


In [2]:
df = pd.read_csv('../../Clusters/cluster_6.csv')
df.columns = df.columns.str.strip()
print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Bankrupt?'].value_counts())
print(f"\nBankruptcy rate: {df['Bankrupt?'].mean()*100:.1f}%")
print("\nFull dataset (key columns):")
print(df[['Index', 'Bankrupt?',
          'ROA(A) before interest and % after tax',
          'Working Capital to Total Assets',
          'Current Liability to Current Assets',
          'Retained Earnings to Total Assets',
          'Current Liability to Assets']].round(4).to_string(index=False))

Shape: (7, 98)

Class distribution:
Bankrupt?
1    5
0    2
Name: count, dtype: int64

Bankruptcy rate: 71.4%

Full dataset (key columns):
 Index  Bankrupt?  ROA(A) before interest and % after tax  Working Capital to Total Assets  Current Liability to Current Assets  Retained Earnings to Total Assets  Current Liability to Assets
   208          1                                  0.2110                           0.7232                               0.0594                             0.7776                       0.1657
  1978          1                                  0.0572                           0.6924                               0.0653                             0.8420                       0.2523
  2468          1                                  0.2236                           0.6431                               0.1939                             0.8440                       0.1602
  2937          0                                  0.2834                           0.0000   

In [3]:
def custom_accuracy(y_true, y_pred):
    """Project metric: TT / (TF + TT) = recall for bankrupt class."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    TT = int(((y_true == 1) & (y_pred == 1)).sum())
    TF = int(((y_true == 1) & (y_pred == 0)).sum())
    if TT + TF == 0:
        return 0.0
    return TT / (TT + TF)

def print_breakdown(y_true, y_pred, label=""):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    TT = int(((y_true==1) & (y_pred==1)).sum())
    TF = int(((y_true==1) & (y_pred==0)).sum())
    FT = int(((y_true==0) & (y_pred==1)).sum())
    FF = int(((y_true==0) & (y_pred==0)).sum())
    acc = custom_accuracy(y_true, y_pred)
    print(f"{'='*40}")
    if label: print(f"  {label}")
    print(f"  TT (bankrupt -> bankrupt): {TT}")
    print(f"  TF (bankrupt -> healthy):  {TF}")
    print(f"  FT (healthy  -> bankrupt): {FT}")
    print(f"  FF (healthy  -> healthy):  {FF}")
    print(f"  Custom Accuracy (TT/(TF+TT)): {acc:.4f}")
    print(f"{'='*40}")

print("Custom metric defined.")

Custom metric defined.


In [4]:
X_all = df.drop(columns=['Index', 'Bankrupt?', 'Cluster'])
y     = df['Bankrupt?'].copy()

SELECTED_FEATURES = [
    'ROA(A) before interest and % after tax',
    'Working Capital to Total Assets',
    'Current Liability to Current Assets',
    'Retained Earnings to Total Assets',
    'Current Liability to Assets',
]
X = X_all[SELECTED_FEATURES].copy()
print(f"Features selected: {len(SELECTED_FEATURES)}")
print(f"Dataset: {X.shape[0]} rows x {X.shape[1]} features")
for f in SELECTED_FEATURES:
    print(f"  - {f}")

Features selected: 5
Dataset: 7 rows x 5 features
  - ROA(A) before interest and % after tax
  - Working Capital to Total Assets
  - Current Liability to Current Assets
  - Retained Earnings to Total Assets
  - Current Liability to Assets


In [5]:
loo = LeaveOneOut()

base_estimators = [
    ('lr',  LogisticRegression(C=0.1, max_iter=2000, random_state=RANDOM_STATE)),
    ('dt',  DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)),
    ('knn', KNeighborsClassifier(n_neighbors=1)),
]

meta_model = LogisticRegression(C=1.0, max_iter=2000, random_state=RANDOM_STATE)

def build_pipeline():
    stacking = StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_model,
        cv=2,
        stack_method='predict_proba',
        passthrough=False,
        n_jobs=1
    )
    return Pipeline([('scaler', StandardScaler()), ('stacking', stacking)])

print("Base models: LR (C=0.1), Decision Stump (depth=1), KNN (k=1)")
print("Meta model:  Logistic Regression (C=1.0)")
print("Inner CV:    StratifiedKFold(2) — safe for 7 rows")

Base models: LR (C=0.1), Decision Stump (depth=1), KNN (k=1)
Meta model:  Logistic Regression (C=1.0)
Inner CV:    StratifiedKFold(2) — safe for 7 rows


In [6]:
loo_preds = np.zeros(len(y), dtype=int)

for train_idx, test_idx in loo.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr        = y.iloc[train_idx]
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)
    votes = []
    for name, est in base_estimators:
        m = clone(est)
        m.fit(X_tr_s, y_tr)
        votes.append(int(m.predict(X_te_s)[0]))
    loo_preds[test_idx] = 1 if sum(votes) >= 2 else 0

print("LOO Cross-Validation (majority vote across 3 base models)\n")
print_breakdown(y, loo_preds, label="LOO Evaluation")
result_df = df[['Index', 'Bankrupt?']].copy()
result_df['LOO_Prediction'] = loo_preds
result_df['Correct'] = (result_df['Bankrupt?'] == result_df['LOO_Prediction'])
print("\nPer-company prediction:")
print(result_df.to_string(index=False))

LOO Cross-Validation (majority vote across 3 base models)

  LOO Evaluation
  TT (bankrupt -> bankrupt): 5
  TF (bankrupt -> healthy):  0
  FT (healthy  -> bankrupt): 2
  FF (healthy  -> healthy):  0
  Custom Accuracy (TT/(TF+TT)): 1.0000

Per-company prediction:
 Index  Bankrupt?  LOO_Prediction  Correct
   208          1               1     True
  1978          1               1     True
  2468          1               1     True
  2937          0               1    False
  3738          1               1     True
  4060          0               1    False
  5052          1               1     True


In [7]:
final_pipeline = build_pipeline()
final_pipeline.fit(X, y)
train_preds = final_pipeline.predict(X)
print("Final model fitted on all 7 training rows.\n")
print_breakdown(y, train_preds, label="Train Accuracy (in-sample)")
print(f"\nSparsity: {train_preds.sum()}/{len(train_preds)} predicted bankrupt ({train_preds.mean()*100:.1f}%)")

Final model fitted on all 7 training rows.

  Train Accuracy (in-sample)
  TT (bankrupt -> bankrupt): 5
  TF (bankrupt -> healthy):  0
  FT (healthy  -> bankrupt): 2
  FF (healthy  -> healthy):  0
  Custom Accuracy (TT/(TF+TT)): 1.0000

Sparsity: 7/7 predicted bankrupt (100.0%)


In [8]:
c6_model = {
    'cluster_id':  6,
    'feature_cols': SELECTED_FEATURES,
    'model':        final_pipeline,
    'threshold':    0.5,
    'n_train':      len(y),
    'n_bankrupt':   int(y.sum()),
    'note':         'LOO stacking — LR + DTree(d=1) + KNN(k=1) -> LR meta'
}
joblib.dump(c6_model, 'c6_stacking_model.joblib')
loaded = joblib.load('c6_stacking_model.joblib')
test_preds = loaded['model'].predict(X[loaded['feature_cols']])
assert list(test_preds) == list(train_preds), "Reload mismatch!"
print("Saved: c6_stacking_model.joblib")
print("Reload test passed.")
print("\nModel info:")
for k, v in c6_model.items():
    if k != 'model':
        print(f"  {k}: {v}")

Saved: c6_stacking_model.joblib
Reload test passed.

Model info:
  cluster_id: 6
  feature_cols: ['ROA(A) before interest and % after tax', 'Working Capital to Total Assets', 'Current Liability to Current Assets', 'Retained Earnings to Total Assets', 'Current Liability to Assets']
  threshold: 0.5
  n_train: 7
  n_bankrupt: 5
  note: LOO stacking — LR + DTree(d=1) + KNN(k=1) -> LR meta
